# NYC Taxi Demand Forecasting — Deep Learning

This notebook evaluates whether a sequence-based neural network can improve one-hour-ahead Yellow Taxi demand forecasts by learning directly from recent demand history.

The existing Random Forest relies on manually engineered lag and rolling features and achieved a December 2025 MAE of 5.72. A Gated Recurrent Unit (GRU) network is evaluated to determine whether representing the previous week of hourly demand as a sequence captures additional temporal structure.

## Objective

1. Construct leakage-safe 168-hour demand sequences for each pickup zone.

2. Train a GRU to forecast demand one hour ahead.

3. Evaluate the model using chronological validation.

4. Compare its performance with the existing Random Forest and previous-week baseline.

5. Determine whether sequence modeling provides enough incremental value to justify its additional complexity.

## 1. Setup

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

processed_dir = Path("../data/processed")

print("TensorFlow version:", tf.__version__)
print("Available devices:", tf.config.list_physical_devices())

TensorFlow version: 2.21.0
Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## 2. Load Hourly Demand

The balanced hourly demand panel is used rather than the previously engineered modeling table because the GRU will receive historical demand sequences directly. Each sequence contains the previous 168 hourly observations for a single pickup zone and predicts demand during the following hour.

In [3]:
demand = pd.read_parquet(
    processed_dir / "yellow_taxi_hourly_demand_2025.parquet"
)

demand = (
    demand
    .sort_values(["PULocationID", "pickup_hour"])
    .reset_index(drop=True)
)

print("Shape:", demand.shape)
print(
    "Date range:",
    demand["pickup_hour"].min(),
    "to",
    demand["pickup_hour"].max()
)
print("Zones:", demand["PULocationID"].nunique())

demand.head()

Shape: (1524240, 3)
Date range: 2025-01-01 00:00:00 to 2025-12-31 23:00:00
Zones: 174


,pickup_hour,PULocationID,trip_count
0,2025-01-01 00:00:00,4,28
1,2025-01-01 01:00:00,4,54
2,2025-01-01 02:00:00,4,52
3,2025-01-01 03:00:00,4,42
4,2025-01-01 04:00:00,4,32


## 3. Construct Historical Demand Sequences

Each prediction uses the previous seven days (168 hours) of demand from the same pickup zone. Sequences are constructed independently within each zone so that observations from different pickup zones are never mixed.

The sequence ending at hour `t-1` is used to predict demand at hour `t`, preserving the one-hour-ahead forecasting boundary.

In [4]:
SEQUENCE_LENGTH = 168

example_zone_id = demand["PULocationID"].iloc[0]

example_zone = (
    demand[demand["PULocationID"] == example_zone_id]
    .sort_values("pickup_hour")
    .reset_index(drop=True)
)

values = example_zone["trip_count"].to_numpy()

X_example = []
y_example = []

for i in range(SEQUENCE_LENGTH, len(values)):
    X_example.append(values[i-SEQUENCE_LENGTH:i])
    y_example.append(values[i])

X_example = np.asarray(X_example, dtype=np.float32)
y_example = np.asarray(y_example, dtype=np.float32)

print("Example zone:", example_zone_id)
print("Original hours:", len(example_zone))
print("Sequences:", X_example.shape)
print("Targets:", y_example.shape)

print("\nFirst sequence:")
print(X_example[0][:10], "...", X_example[0][-10:])

print("\nFirst target:")
print(y_example[0])

Example zone: 4
Original hours: 8760
Sequences: (8592, 168)
Targets: (8592,)

First sequence:
[28. 54. 52. 42. 32.  5.  5.  3.  1.  2.] ... [2. 0. 3. 4. 1. 3. 0. 4. 2. 0.]

First target:
3.0


## 4. Prepare the Global Sequence Dataset

A single global model is trained across all pickup zones. This allows the neural network to learn temporal patterns shared across the city while retaining information about which zone generated each sequence.

Rather than training a separate model for every location, each pickup zone is mapped to an integer identifier that can later be represented using a learned embedding.

Sequences are generated independently within each zone using the previous 168 hours of demand.

In [5]:
zone_ids = sorted(demand["PULocationID"].unique())

zone_to_index = {
    zone_id: idx
    for idx, zone_id in enumerate(zone_ids)
}

demand["zone_index"] = (
    demand["PULocationID"]
    .map(zone_to_index)
    .astype(np.int32)
)

print("Number of zones:", len(zone_ids))
print("Zone index range:",
      demand["zone_index"].min(),
      "to",
      demand["zone_index"].max())

Number of zones: 174
Zone index range: 0 to 173


## 5. Define Chronological Splits

The neural network is developed using chronological data splits. Training examples have target timestamps through September 2025, while October and November are reserved for validation.

December was already examined during earlier model development and is therefore not used to make neural-network architecture or training decisions.

In [6]:
TRAIN_END = pd.Timestamp("2025-09-30 23:00:00")
VAL_START = pd.Timestamp("2025-10-01 00:00:00")
VAL_END = pd.Timestamp("2025-11-30 23:00:00")
DEC_START = pd.Timestamp("2025-12-01 00:00:00")

print("Training through:", TRAIN_END)
print("Validation:", VAL_START, "through", VAL_END)
print("December benchmark begins:", DEC_START)

Training through: 2025-09-30 23:00:00
Validation: 2025-10-01 00:00:00 through 2025-11-30 23:00:00
December benchmark begins: 2025-12-01 00:00:00


## 6. Normalize Demand Using Training Data

Taxi demand varies substantially across pickup zones. Neural networks generally train more reliably when inputs are placed on comparable scales, so demand is standardized separately within each zone.

Means and standard deviations are calculated using only the training period to prevent information from later months from influencing preprocessing.

In [7]:
train_stats = (
    demand[demand["pickup_hour"] <= TRAIN_END]
    .groupby("PULocationID")["trip_count"]
    .agg(["mean", "std"])
    .rename(columns={
        "mean": "train_mean",
        "std": "train_std"
    })
)

demand = demand.merge(
    train_stats,
    on="PULocationID",
    how="left"
)

# Defensive handling in case a zone has zero variance.
demand["train_std"] = demand["train_std"].replace(0, 1)

demand["demand_scaled"] = (
    (demand["trip_count"] - demand["train_mean"])
    / demand["train_std"]
).astype(np.float32)

print(
    demand[
        ["train_mean", "train_std", "demand_scaled"]
    ].describe()
)

         train_mean     train_std  demand_scaled
count  1.524240e+06  1.524240e+06   1.524240e+06
mean   3.124240e+01  2.406971e+01   1.938220e-02
std    5.288378e+01  3.992297e+01   1.013485e+00
min    8.302808e-01  1.105714e+00  -1.828732e+00
25%    1.634615e+00  1.737082e+00  -7.721729e-01
50%    4.840812e+00  4.308908e+00  -1.708078e-01
75%    3.527717e+01  2.488912e+01   5.539462e-01
max    2.368964e+02  2.043325e+02   2.287958e+01


## 7. Create Sequence Indices

Materializing every 168-hour sequence would duplicate a large amount of data in memory. Instead, the underlying zone histories are stored once and training examples are represented by their target positions.

Sequences are extracted dynamically during training.

In [8]:
zone_arrays = {}

train_samples = []
val_samples = []
dec_samples = []

for zone_id in zone_ids:
    zone_df = (
        demand[demand["PULocationID"] == zone_id]
        .sort_values("pickup_hour")
        .reset_index(drop=True)
    )

    values = zone_df["demand_scaled"].to_numpy(dtype=np.float32)
    targets = zone_df["trip_count"].to_numpy(dtype=np.float32)
    timestamps = zone_df["pickup_hour"].to_numpy()
    zone_index = zone_to_index[zone_id]

    zone_arrays[zone_index] = {
        "values": values,
        "targets": targets,
        "timestamps": timestamps,
        "mean": float(zone_df["train_mean"].iloc[0]),
        "std": float(zone_df["train_std"].iloc[0])
    }

    for i in range(SEQUENCE_LENGTH, len(zone_df)):
        timestamp = pd.Timestamp(timestamps[i])
        sample = (zone_index, i)

        if timestamp <= TRAIN_END:
            train_samples.append(sample)

        elif VAL_START <= timestamp <= VAL_END:
            val_samples.append(sample)

        elif timestamp >= DEC_START:
            dec_samples.append(sample)

print("Training samples:", len(train_samples))
print("Validation samples:", len(val_samples))
print("December benchmark samples:", len(dec_samples))

Training samples: 1110816
Validation samples: 254736
December benchmark samples: 129456


## 8. Build Batched Sequence Datasets

Training sequences are generated in batches rather than materializing the full three-dimensional sequence dataset in memory. Each batch contains a 168-hour standardized demand history and the corresponding pickup-zone identifier.

The target remains in its original trip-count scale so model performance can be evaluated directly using MAE.

In [9]:
class TaxiSequence(keras.utils.Sequence):
    def __init__(
        self,
        samples,
        zone_arrays,
        batch_size=1024,
        shuffle=False,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.samples = np.asarray(samples, dtype=np.int32)
        self.zone_arrays = zone_arrays
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.samples))

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, batch_idx):
        batch_indices = self.indices[
            batch_idx * self.batch_size:
            (batch_idx + 1) * self.batch_size
        ]

        batch_samples = self.samples[batch_indices]
        n = len(batch_samples)

        X_seq = np.empty(
            (n, SEQUENCE_LENGTH, 1),
            dtype=np.float32
        )

        X_zone = np.empty(
            (n,),
            dtype=np.int32
        )

        y = np.empty(
            (n,),
            dtype=np.float32
        )

        for j, (zone_index, target_idx) in enumerate(batch_samples):
            zone_data = self.zone_arrays[int(zone_index)]

            X_seq[j, :, 0] = zone_data["values"][
                target_idx - SEQUENCE_LENGTH:target_idx
            ]

            X_zone[j] = zone_index
            y[j] = zone_data["targets"][target_idx]

        return {
            "sequence": X_seq,
            "zone": X_zone
        }, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [10]:
BATCH_SIZE = 1024

train_dataset = TaxiSequence(
    train_samples,
    zone_arrays,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataset = TaxiSequence(
    val_samples,
    zone_arrays,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_dataset))
print("Validation batches:", len(val_dataset))

batch_x, batch_y = train_dataset[0]

print("Sequence batch:", batch_x["sequence"].shape)
print("Zone batch:", batch_x["zone"].shape)
print("Target batch:", batch_y.shape)

Training batches: 1085
Validation batches: 249
Sequence batch: (1024, 168, 1)
Zone batch: (1024,)
Target batch: (1024,)


## 9. Build the GRU Forecasting Model

The model combines two sources of information:

- a GRU representation of the previous 168 hours of standardized demand;
- a learned embedding representing the pickup zone.

The zone embedding allows the network to learn location-specific characteristics without imposing an artificial numeric ordering on TLC zone identifiers. A relatively small network is used to limit unnecessary complexity and keep training computationally practical.

In [11]:
NUM_ZONES = len(zone_ids)
EMBEDDING_DIM = 8

sequence_input = keras.Input(
    shape=(SEQUENCE_LENGTH, 1),
    name="sequence"
)

zone_input = keras.Input(
    shape=(),
    dtype=tf.int32,
    name="zone"
)

sequence_features = layers.GRU(
    32,
    name="gru"
)(sequence_input)

zone_embedding = layers.Embedding(
    input_dim=NUM_ZONES,
    output_dim=EMBEDDING_DIM,
    name="zone_embedding"
)(zone_input)

zone_embedding = layers.Flatten()(zone_embedding)

combined = layers.Concatenate()([
    sequence_features,
    zone_embedding
])

x = layers.Dense(
    32,
    activation="relu"
)(combined)

output = layers.Dense(
    1,
    activation="relu",
    name="demand"
)(x)

gru_model = keras.Model(
    inputs={
        "sequence": sequence_input,
        "zone": zone_input
    },
    outputs=output
)

gru_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

gru_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ zone (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequence            │ (None, 168, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zone_embedding      │ (None, 8)         │      1,392 │ zone[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 32)        │      3,360 │ sequence[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ zone_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 40)        │          0 │ gru[0][0],        │
│ (Concatenate)       │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      1,312 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ demand (Dense)      │ (None, 1)         │         33 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,097 (23.82 KB)

 Trainable params: 6,097 (23.82 KB)

 Non-trainable params: 0 (0.00 B)

## 10. Train the GRU

The network is trained using mean squared error as the optimization loss while MAE is monitored as the primary evaluation metric. Early stopping monitors validation MAE and restores the weights from the best epoch, reducing unnecessary training and limiting overfitting.

In [12]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_mae",
        patience=3,
        restore_best_weights=True,
        mode="min"
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_mae",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        mode="min"
    )
]

history = gru_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 164s 151ms/step - loss: 1928.3788 - mae: 18.2363 - val_loss: 563.9970 - val_mae: 10.6795 - learning_rate: 0.0010
Epoch 2/20
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 166s 153ms/step - loss: 387.4388 - mae: 8.7034 - val_loss: 335.1339 - val_mae: 7.9222 - learning_rate: 0.0010
Epoch 3/20
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 164s 151ms/step - loss: 293.9863 - mae: 7.4696 - val_loss: 297.2894 - val_mae: 7.3939 - learning_rate: 0.0010
Epoch 4/20
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 196s 181ms/step - loss: 268.2317 - mae: 7.0928 - val_loss: 279.7866 - val_mae: 7.1072 - learning_rate: 0.0010
Epoch 5/20
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 186s 172ms/step - loss: 255.6465 - mae: 6.8881 - val_loss: 270.1513 - val_mae: 6.9582 - learning_rate: 0.0010
Epoch 6/20
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 176s 162ms/step - loss: 242.8461 - mae: 6.7061 - val_loss: 242.2500 - val_mae: 6.7144 - learning_rate: 0.0010
Epoch 7/20
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 183s 169ms/step - loss: 212.3579 - mae: 6.2